
6.2 Read Bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "ask_sarthak.bronze.gl_transactions_raw"
)

print("Bronze records:", bronze_df.count())

bronze_df.printSchema()


6.3 Standardize data types

In [0]:
silver_stage = (
    bronze_df
    .withColumn("company_code", F.trim(F.col("company_code").cast("string")))
    .withColumn("document_number", F.trim(F.col("document_number").cast("string")))
    .withColumn("gl_account", F.trim(F.col("gl_account").cast("string")))
    .withColumn("debit_credit", F.upper(F.trim(F.col("debit_credit"))))
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("amount", F.col("amount").cast("decimal(18,2)"))
    .withColumn("posting_date", F.to_date("posting_date"))
    .withColumn("document_date", F.to_date("document_date"))
    .withColumn("fiscal_year", F.col("fiscal_year").cast("int"))
    .withColumn("line_item", F.trim(F.col("line_item").cast("string")))
)

6.4 Identify invalid records

In [0]:
invalid_condition = (
    F.col("company_code").isNull()
    | F.col("document_number").isNull()
    | F.col("gl_account").isNull()
    | F.col("posting_date").isNull()
    | F.col("amount").isNull()
    | (F.col("amount") <= 0)
    | (~F.col("debit_credit").isin("D", "C"))
    | F.col("currency").isNull()
)

invalid_df = silver_stage.filter(invalid_condition)
valid_df = silver_stage.filter(~invalid_condition)

print("Valid records:", valid_df.count())
print("Invalid records:", invalid_df.count())


6.5 Detect duplicate journal lines

In [0]:
duplicate_window = (
    Window
    .partitionBy(
        "company_code",
        "fiscal_year",
        "document_number",
        "line_item"
    )
    .orderBy(F.col("_ingestion_timestamp").desc())
)

ranked_df = valid_df.withColumn(
    "_row_number",
    F.row_number().over(duplicate_window)
)

duplicates_df = (
    ranked_df
    .filter(F.col("_row_number") > 1)
    .drop("_row_number")
)

silver_df = (
    ranked_df
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print("Duplicate records:", duplicates_df.count())
print("Clean records:", silver_df.count())


6.6 Save the Silver tables

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ask_sarthak.silver.gl_transactions")
)

(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ask_sarthak.silver.gl_invalid_records")
)

(
    duplicates_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ask_sarthak.silver.gl_duplicate_records")
)


6.7 Validate the Silver layer

In [0]:
display(
    spark.sql("""
        SELECT
            'Bronze' AS layer,
            COUNT(*) AS total_records
        FROM ask_sarthak.bronze.gl_transactions_raw

        UNION ALL

        SELECT
            'Silver' AS layer,
            COUNT(*) AS total_records
        FROM ask_sarthak.silver.gl_transactions

        UNION ALL

        SELECT
            'Invalid' AS layer,
            COUNT(*) AS total_records
        FROM ask_sarthak.silver.gl_invalid_records

        UNION ALL

        SELECT
            'Duplicates' AS layer,
            COUNT(*) AS total_records
        FROM ask_sarthak.silver.gl_duplicate_records
    """)
)


Step 6.8 — Check journal balance

In [0]:
display(
    spark.sql("""
        SELECT
            company_code,
            fiscal_year,
            document_number,
            SUM(CASE
                WHEN debit_credit = 'D' THEN amount
                ELSE 0
            END) AS total_debit,
            SUM(CASE
                WHEN debit_credit = 'C' THEN amount
                ELSE 0
            END) AS total_credit
        FROM ask_sarthak.silver.gl_transactions
        GROUP BY
            company_code,
            fiscal_year,
            document_number
        HAVING total_debit <> total_credit
        LIMIT 20
    """)
)